In [1]:
import pandas as pd

df = pd.read_csv("QACP_Python_QA_ar_with_model_answers.csv", encoding="utf-8-sig")
df.head()
df.columns

Index(['id', 'Knowledge Point', 'Question Type', 'Answer Type', 'question_ar',
       'answer_ar', 'answer_model_ar'],
      dtype='object')

In [2]:
import os
from openai import OpenAI
import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [8]:
JUDGE_SYSTEM_PROMPT = """
You are a fair and strict Python teaching assistant.

You evaluate a student's Arabic answer to a Python question.

Rules:
- Focus on correctness and completeness of the Python concepts.
- Use the expert reference answer as a guide to what a good answer should cover,
  but accept different wording or structure.
- Do NOT penalize answers just because they are longer or shorter.
- Do NOT require the student's answer to match the reference wording.
- Only penalize if the student's answer is missing important ideas,
  contains incorrect information, or is confusing for the learner.

Scoring (1 to 3):
- 3: Fully correct and complete; good explanation for this learner level.
- 2: Partially correct; important points missing or some confusion, but acceptable.
- 1: Completely incorrect or off-topic.

Output format:
- Respond with ONLY a single integer from 1 to 3.
"""

In [9]:
def judge_answer(question_ar: str,
                 expert_answer_ar: str,
                 model_answer_ar: str,
                 question_type: str,
                 model: str = "gpt-5-mini") -> int:
    """
    Use gpt-5-mini to judge the model's answer against the expert answer.
    Returns an integer score in [1..3].
    """
    if not isinstance(question_ar, str) or question_ar.strip() == "":
        return 0
    if not isinstance(model_answer_ar, str) or model_answer_ar.strip() == "":
        return 1  # empty or missing answer is basically very poor

    qt = question_type if isinstance(question_type, str) else "Beginner"

    user_content = f"""
نوع المتعلّم: {qt}

السؤال (باللغة العربية):
{question_ar}

الإجابة المرجعية (من الخبير، باللغة العربية):
{expert_answer_ar}

إجابة النموذج (التي نريد تقييمها، باللغة العربية):
{model_answer_ar}

قيّم إجابة النموذج حسب المقياس من 1 إلى 3 المذكور في التعليمات.
تذكَّر:
- لا تعاقب الاختلاف في الصياغة أو في الطول إذا كانت الفكرة صحيحة.
- ركّز على مدى صحة المفاهيم في بايثون، واكتمال الشرح.
"""

    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ]
    )

    raw = resp.choices[0].message.content.strip()

    # Try to parse as an integer directly
    try:
        score = int(raw)
        if score < 1 or score > 3:
            raise ValueError
        return score
    except Exception:
        # Fallback: look for the first digit 1-3 in the response
        for ch in raw:
            if ch in "123":
                return int(ch)
        # If all fails, return 0 to indicate parsing problem
        return 0

In [10]:
sample_row = df.iloc[0]

print("=== Question Type ===")
print(sample_row["Question Type"])

print("\n=== Question (Arabic) ===")
print(sample_row["question_ar"])

print("\n=== Expert Answer (Arabic) ===")
print(sample_row["answer_ar"][:400])

print("\n=== Model Answer (Arabic) ===")
print(sample_row["answer_model_ar"][:400])

score_example = judge_answer(
    sample_row["question_ar"],
    sample_row["answer_ar"],
    sample_row["answer_model_ar"],
    sample_row["Question Type"]
)

print("\nJudge score for this example:", score_example)

=== Question Type ===
Experience in Programming

=== Question (Arabic) ===
ما هو بايثون؟

=== Expert Answer (Arabic) ===
لغة بايثون منذ ولادتها في عام 1991 وحتى اليوم بلغت حوالي 30 عاماً، في السنوات الأخيرة ومع التطور السريع للإنترنت وظهور صناعات مثل الذكاء الاصطناعي وإنترنت الأشياء وتحليل البيانات، أصبحت بايثون كواحدة من لغات الدعم الهامة أكثر شيوعاً، وازدادت شعبيتها باستمرار. مجالات استخدام بايثون واسعة جداً، سواء في التشغيل والصيانة، والهندسة المعمارية، وتطوير الويب، وتحليل البيانات، أو اختبار الخوارزميات، لديها 

=== Model Answer (Arabic) ===
بايثون لغة برمجة عالية المستوى، مفسَّرة وتquential وتستهدف كتابة الشفرة بوضوح وبساطة. صممت لتكونReadable، وتسمح لك بإنجاز المهام بشكل أسرع مع تقليل التعقيد. تَستخدم بايثون غلافاً واسعاً من المكتبات والإطارات، وتعمل عبر منصات متعددة (Windows، macOS، Linux).

أهم ما يميز بايثون
- سهولة القراءة والكتابة: بنية الشفرة واضحة وتستخدم المسافات البادئة (Indentation) لتنظيم الكود بدلاً من علامات التبويب الم

Judge score for this example: 3


In [5]:
sample_row = df.iloc[0]

print("=== Question Type ===")
print(sample_row["Question Type"])

print("\n=== Question (Arabic) ===")
print(sample_row["question_ar"])

print("\n=== Expert Answer (Arabic) ===")
print(sample_row["answer_ar"][:400])

print("\n=== Model Answer (Arabic) ===")
print(sample_row["answer_model_ar"][:400])

score_example = judge_answer(
    sample_row["question_ar"],
    sample_row["answer_ar"],
    sample_row["answer_model_ar"],
    sample_row["Question Type"]
)

print("\nJudge score for this example:", score_example)

=== Question Type ===
Experience in Programming

=== Question (Arabic) ===
ما هو بايثون؟

=== Expert Answer (Arabic) ===
لغة بايثون منذ ولادتها في عام 1991 وحتى اليوم بلغت حوالي 30 عاماً، في السنوات الأخيرة ومع التطور السريع للإنترنت وظهور صناعات مثل الذكاء الاصطناعي وإنترنت الأشياء وتحليل البيانات، أصبحت بايثون كواحدة من لغات الدعم الهامة أكثر شيوعاً، وازدادت شعبيتها باستمرار. مجالات استخدام بايثون واسعة جداً، سواء في التشغيل والصيانة، والهندسة المعمارية، وتطوير الويب، وتحليل البيانات، أو اختبار الخوارزميات، لديها 

=== Model Answer (Arabic) ===
بايثون لغة برمجة عالية المستوى، مفسَّرة وتquential وتستهدف كتابة الشفرة بوضوح وبساطة. صممت لتكونReadable، وتسمح لك بإنجاز المهام بشكل أسرع مع تقليل التعقيد. تَستخدم بايثون غلافاً واسعاً من المكتبات والإطارات، وتعمل عبر منصات متعددة (Windows، macOS، Linux).

أهم ما يميز بايثون
- سهولة القراءة والكتابة: بنية الشفرة واضحة وتستخدم المسافات البادئة (Indentation) لتنظيم الكود بدلاً من علامات التبويب الم

Judge score for this example: 5


In [6]:
for i in range(1, 5):  # rows 1 to 4 inclusive
    sample_row = df.iloc[i]

    print("\n" + "=" * 80)
    print(f"Row index: {i}")

    print("=== Question Type ===")
    print(sample_row["Question Type"])

    print("\n=== Question (Arabic) ===")
    print(sample_row["question_ar"])

    print("\n=== Expert Answer (Arabic) ===")
    print(sample_row["answer_ar"][:400])

    print("\n=== Model Answer (Arabic) ===")
    print(sample_row["answer_model_ar"][:400])

    score_example = judge_answer(
        sample_row["question_ar"],
        sample_row["answer_ar"],
        sample_row["answer_model_ar"],
        sample_row["Question Type"]
    )

    print("\nJudge score for this example:", score_example)



Row index: 1
=== Question Type ===
Beginner

=== Question (Arabic) ===
ما هو بايثون؟

=== Expert Answer (Arabic) ===
## ما هو بايثون؟
بايثون هي لغة برمجة شهيرة جدًا، وكذلك كما يتواصل الناس فيما بينهم باستخدام اللغة الصينية، بايثون هي اللغة التي تخاطب بها الإنسان الحاسوب، من خلالها تخبر الحاسوب بما يجب أن يفعله وكيفية فعله. من المواقع والخدمات المعروفة محليًا وعالميًا التي نستخدمها يوميًا، مثل YouTube وInstagram وToutiao وZhihu وغيرها، غالبًا ما تكون خلفياتها (الجهة الخلفية) مطوّرة باستخدام بايثون.

# ما الذي يمك

=== Model Answer (Arabic) ===
بايثون هي لغة برمجة عالية المستوى ومفسَّرة، مصممة لتكون سهلة القراءة والكتابة. صممت لكي يشعر المبرمج بالمتعة خلال كتابة الكود وتطويره بسرعة، دون تعقيد زائد.

نقاط رئيسية عن بايثون:
- متعددة الأنماط: تدعم البرمجة الإجرائية، والكائنية (object-oriented)، والدوالية (functional).
- ديناميكية typing: لا تحتاج إلى إعلان نوع المتغير صراحةً، لكن أنواعها تتغير أثناء التشغيل.
- إدارة الذاكرة تلقائياً: تجمع ا

Judge score for this example: 5

Row index: 2
==

In [7]:
for i in range(125, 133):
    sample_row = df.iloc[i]

    print("\n" + "=" * 80)
    print(f"Row index: {i}")

    print("=== Question Type ===")
    print(sample_row["Question Type"])

    print("\n=== Question (Arabic) ===")
    print(sample_row["question_ar"])

    print("\n=== Expert Answer (Arabic) ===")
    print(sample_row["answer_ar"][:400])

    print("\n=== Model Answer (Arabic) ===")
    print(sample_row["answer_model_ar"][:400])

    score_example = judge_answer(
        sample_row["question_ar"],
        sample_row["answer_ar"],
        sample_row["answer_model_ar"],
        sample_row["Question Type"]
    )

    print("\nJudge score for this example:", score_example)


Row index: 125
=== Question Type ===
Experience in Programming

=== Question (Arabic) ===
كيف يمكن تحويل حلقة for إلى قائمة باستخدام فهم القائمة؟

=== Expert Answer (Arabic) ===
قوائم التوليد في بايثون هي بنية نحوية فريدة من نوعها، وهي توفر طريقة مختصرة وسريعة لإنشاء القوائم. يمكن لقوائم التوليد في بايثون أن تحل محل العديد من الحالات التي تستخدم الحلقات والشروط، وتكون كفاءتها أعلى، كما أن الكود أكثر وضوحاً وأسهل قراءة. فيما يلي سأوضح بمثال كيف نقوم بتحويل حلقة for إلى بناء القائمة باستخدام قوائم التوليد.

### المثال: حساب مربعات كل رقم في القائمة

#### باستخدام حلقة for



=== Model Answer (Arabic) ===
باختصار: تحويل حلقة for إلى قائمة باستخدام فهم القائمة (list comprehension) يعني بناء القائمة مباشرة من تعبير واحد، بدلاً من إنشاءها ثم استدعاء append في كل تكرار.

الصيغة الأساسية
- [expression for item in iterable if condition]
  - expression: ما تريد وضعه في كل عنصر من الناتج
  - item: اسم المتغير الذي يستلم كل عنصر
  - iterable: المجموعة التي تتكرر عليها
  - if condition: شرط اختيار

In [11]:
df["score"] = 0  # initialize

for idx, row in df.iterrows():
    q_ar = row["question_ar"]
    a_expert = row["answer_ar"]
    a_model = row["answer_model_ar"]
    q_type = row["Question Type"]

    df.at[idx, "score"] = judge_answer(q_ar, a_expert, a_model, q_type)

    if (idx + 1) % 20 == 0:
        print(f"Judged {idx + 1} answers...")

Judged 20 answers...
Judged 40 answers...
Judged 60 answers...
Judged 80 answers...
Judged 100 answers...
Judged 120 answers...
Judged 140 answers...
Judged 160 answers...
Judged 180 answers...
Judged 200 answers...
Judged 220 answers...
Judged 240 answers...
Judged 260 answers...
Judged 280 answers...
Judged 300 answers...
Judged 320 answers...
Judged 340 answers...
Judged 360 answers...
Judged 380 answers...
Judged 400 answers...
Judged 420 answers...
Judged 440 answers...
Judged 460 answers...
Judged 480 answers...
Judged 500 answers...
Judged 520 answers...


In [12]:
output_path = "QACP_Python_QA_ar_with_model_answers_scored.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print("Saved to:", output_path)

Saved to: QACP_Python_QA_ar_with_model_answers_scored.csv


In [14]:
counts = df["score"].value_counts().sort_index()
total = counts.sum()

for score, n in counts.items():
    pct = (n / total) * 100
    print(f"{score} = {n} rows, {pct:.1f}%")

1 = 2 rows, 0.4%
2 = 36 rows, 6.7%
3 = 496 rows, 92.9%


In [15]:
for s in [1, 2]:
    idx = df.index[df["score"] == s].tolist()
    print(f"{s} = rows {', '.join(map(str, idx)) if idx else '(none)'}")

1 = rows 307, 394
2 = rows 16, 72, 88, 110, 128, 138, 144, 158, 165, 168, 170, 194, 200, 231, 238, 263, 281, 282, 284, 299, 306, 314, 316, 367, 377, 386, 430, 431, 432, 441, 442, 455, 456, 484, 499, 524
